# Backtesting Your Strategy with SportsQuant

**Prove your strategy works before risking money**

A strategy that looks great in hindsight often fails in practice. Backtesting lets you simulate how a strategy would have performed on historical data. Walk-forward validation goes further — it prevents look-ahead bias and overfitting by using only past data for each prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.engine import decide_over_under
from sportsquant.core.betting.kelly import KellyCalculator
from sportsquant.core.betting.metrics import (
    calculate_performance_metrics,
    calculate_sharpe_ratio,
    calculate_max_drawdown,
    calculate_profit_factor,
)
from sportsquant.core.backtest.regime import (
    WalkForwardBacktest,
    WalkForwardConfig,
    RegimePeriod,
    SensitivityAnalyzer,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Imports successful. Let's backtest!")

## 1. Why Backtesting Matters

Common pitfalls that backtesting helps you avoid:

| Pitfall | Description | How to Fix |
|---------|-------------|------------|
| **Overfitting** | Fitting parameters to noise in historical data | Walk-forward validation |
| **Look-ahead bias** | Using future data in past predictions | Strict temporal splits |
| **Survivorship bias** | Only testing on players/games that existed | Include all available data |
| **Selection bias** | Cherry-picking favorable periods | Test across all regimes |

## 2. Create a Simple Value Betting Strategy

We'll define a strategy: **Bet OVER when our model's probability exceeds implied probability by at least 2% (edge threshold).**

In [ ]:
np.random.seed(42)

# Generate synthetic historical data: 200 NBA games with props
n_games = 200

historical_data = pd.DataFrame({
    "game_date": pd.date_range("2024-10-22", periods=n_games, freq="D"),
    "player_id": np.random.choice(range(1, 31), n_games),
    "player_name": np.random.choice(
        ["LeBron", "Curry", "Doncic", "Jokic", "Giannis", "Tatum", "Embiid", "SGA",
         "Durant", "Davis", "Young", "Lillard", "Morant", "Booker", "Mitchell",
         "Harden", "Butler", "Sabonis", "Fox", "Edwards", "Brunson", "Haliburton",
         "Markkanen", "Ingram", "LaVine", "Murray", "Trae", "Cade", "Wemby", "Bane"],
        n_games
    ),
    "stat": np.random.choice(["Points", "Rebounds", "Assists"], n_games, p=[0.5, 0.25, 0.25]),
    "line": np.random.normal(20, 8, n_games).round(0.5),
    "odds_american_over": np.random.choice([-110, -115, -105, -108], n_games),
    "odds_american_under": np.random.choice([-110, -105, -115, -112], n_games),
})

# Our model's estimated probability (slightly better than random due to "skill")
historical_data["model_prob_over"] = np.clip(
    np.random.beta(52, 48, n_games), 0.35, 0.70
)

# Actual outcome: correlated with model but with noise
noise = np.random.normal(0, 0.05, n_games)
historical_data["actual_prob_over"] = np.clip(historical_data["model_prob_over"] + noise, 0.01, 0.99)
historical_data["actual_over"] = (np.random.random(n_games) < historical_data["actual_prob_over"]).astype(int)

# Compute edge and implied probability
historical_data["implied_prob"] = historical_data["odds_american_over"].apply(
    lambda x: Odds(american=x).implied_prob()
)
historical_data["edge"] = historical_data["model_prob_over"] - historical_data["implied_prob"]

print(f"Generated {len(historical_data)} historical prop lines")
print(f"Average model edge: {historical_data['edge'].mean():.4f}")
print(f"Positive edge rate: {(historical_data['edge'] > 0).mean():.1%}")
historical_data.head()

## 3. Basic Backtest: ROI, Win Rate, Sharpe

Run a simple backtest: bet on every prop where edge > 2%.

In [ ]:
edge_threshold = 0.02

def run_basic_backtest(data, threshold):
    """Run a simple backtest with an edge threshold."""
    bets = data[data["edge"] > threshold].copy()
    
    pnls = []
    evs = []
    outcomes = []
    stakes = []
    
    for _, row in bets.iterrows():
        odds_over = Odds(american=int(row["odds_american_over"]))
        odds_under = Odds(american=int(row["odds_american_under"]))
        
        decision = decide_over_under(
            line=row["line"],
            p_over=row["model_prob_over"],
            odds_over=odds_over,
            odds_under=odds_under,
            true_prob_over=row["model_prob_over"],
            true_prob_under=1.0 - row["model_prob_over"],
        )
        
        stake = 1.0  # unit stakes
        win = (decision.side == "over" and row["actual_over"] == 1) or \
              (decision.side == "under" and row["actual_over"] == 0)
        
        pnl = (decision.decimal_odds - 1) * stake if win else -stake
        pnls.append(pnl)
        evs.append(decision.ev)
        outcomes.append(win)
        stakes.append(stake)
    
    if not pnls:
        return None
    
    metrics = calculate_performance_metrics(
        pnl_series=pd.Series(pnls),
        ev_series=pd.Series(evs),
        outcome_series=pd.Series(outcomes),
        stake_series=pd.Series(stakes),
    )
    
    return metrics, pnls, outcomes

result = run_basic_backtest(historical_data, edge_threshold)
if result:
    metrics, pnls, outcomes = result
    print(f"=== Basic Backtest (edge > {edge_threshold:.0%}) ===")
    print(f"Total bets: {metrics.total_bets}")
    print(f"Win rate: {metrics.win_rate:.1%}")
    print(f"Total P&L: ${metrics.total_pnl:.2f} per $1 unit")
    print(f"Sharpe ratio: {metrics.sharpe_ratio:.2f}")
    print(f"Sortino ratio: {metrics.sortino_ratio:.2f}")
    print(f"Max drawdown: {metrics.max_drawdown:.2f}")
    print(f"Profit factor: {metrics.profit_factor:.2f}")
    print(f"Max win streak: {metrics.max_win_streak}")
    print(f"Max loss streak: {metrics.max_loss_streak}")

In [ ]:
# Visualize equity curve
if result:
    cumulative = np.cumsum(pnls)
    running_max = np.maximum.accumulate(cumulative)
    drawdown = running_max - cumulative
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    # Equity curve
    axes[0].plot(cumulative, color="steelblue", linewidth=2)
    axes[0].fill_between(range(len(cumulative)), 0, cumulative, where=cumulative > 0, alpha=0.2, color="green")
    axes[0].fill_between(range(len(cumulative)), 0, cumulative, where=cumulative < 0, alpha=0.2, color="red")
    axes[0].set_ylabel("Cumulative P&L ($)")
    axes[0].set_title("Equity Curve")
    
    # Drawdown
    axes[1].fill_between(range(len(drawdown)), 0, drawdown, color="red", alpha=0.3)
    axes[1].set_ylabel("Drawdown ($)")
    axes[1].set_title("Drawdown")
    
    # Rolling Sharpe (20-bet window)
    pnl_arr = np.array(pnls)
    window = 20
    rolling_sharpe = []
    for i in range(len(pnl_arr)):
        start = max(0, i - window)
        window_pnl = pnl_arr[start:i+1]
        if len(window_pnl) > 2 and np.std(window_pnl) > 0:
            rolling_sharpe.append(np.mean(window_pnl) / np.std(window_pnl) * np.sqrt(252))
        else:
            rolling_sharpe.append(0)
    
    axes[2].plot(rolling_sharpe, color="purple", linewidth=1.5)
    axes[2].axhline(y=0, color="black", linestyle="--", alpha=0.5)
    axes[2].set_ylabel("Rolling Sharpe (20-bet)")
    axes[2].set_xlabel("Bet Number")
    axes[2].set_title("Rolling Sharpe Ratio")
    
    plt.tight_layout()
    plt.show()

## 4. Walk-Forward Validation

Walk-forward validation is the **gold standard**. Instead of fitting parameters on all data, it:
1. Trains on the first N games (e.g., 82)
2. Tests on the next M games (e.g., 20)
3. Expands the training window and repeats

This mimics real-world deployment: you can only use *past* data to predict *future* outcomes.

In [ ]:
# Prepare features for walk-forward
from sklearn.ensemble import GradientBoostingClassifier

# Build feature matrix from historical data
feature_cols = ["model_prob_over", "edge", "implied_prob", "line"]
X = historical_data[feature_cols].fillna(0).copy()
y = historical_data["actual_over"]

# Run walk-forward backtest
wfb = WalkForwardBacktest(
    features=X,
    y=y,
    train_window=82,
    test_window=20,
)

results = wfb.run(
    model_factory=lambda: GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
    feature_names=feature_cols,
)

print(f"Walk-Forward Results: {results.total_folds} folds")
print(f"\nOverall metrics:")
for k, v in results.overall_metrics.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Compare walk-forward vs naive backtest
fold_accuracies = []
fold_log_losses = []

for fold in results.fold_results:
    if "error" not in fold.metrics:
        fold_accuracies.append(fold.metrics.get("accuracy", 0))
        fold_log_losses.append(fold.metrics.get("log_loss", 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by fold
axes[0].bar(range(len(fold_accuracies)), fold_accuracies, color="steelblue", alpha=0.7)
axes[0].axhline(y=0.5, color="red", linestyle="--", label="Random (50%)")
axes[0].set_xlabel("Fold")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Walk-Forward: Accuracy by Fold")
axes[0].legend()

# Log loss by fold
axes[1].bar(range(len(fold_log_losses)), fold_log_losses, color="coral", alpha=0.7)
axes[1].axhline(y=-np.log(0.5), color="red", linestyle="--", label="Random baseline")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Log Loss")
axes[1].set_title("Walk-Forward: Log Loss by Fold (lower is better)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Sensitivity Analysis: Finding Optimal Edge Threshold

The `SensitivityAnalyzer` sweeps parameter values to find the optimal configuration.

In [ ]:
# Sensitivity analysis on edge threshold
thresholds = [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
sensitivity_results = []

for thresh in thresholds:
    result = run_basic_backtest(historical_data, thresh)
    if result:
        m, _, _ = result
        sensitivity_results.append({
            "threshold": thresh,
            "n_bets": m.total_bets,
            "win_rate": m.win_rate,
            "total_pnl": m.total_pnl,
            "sharpe": m.sharpe_ratio,
            "profit_factor": m.profit_factor,
            "max_drawdown": m.max_drawdown,
        })

sens_df = pd.DataFrame(sensitivity_results)
sens_df.style.format({
    "threshold": "{:.2f}", "win_rate": "{:.1%}",
    "total_pnl": "{:+.2f}", "sharpe": "{:.2f}", "profit_factor": "{:.2f}", "max_drawdown": "{:.2f}"
})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# P&L vs threshold
axes[0].plot(sens_df["threshold"], sens_df["total_pnl"], "o-", color="steelblue", linewidth=2)
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Edge Threshold")
axes[0].set_ylabel("Total P&L ($)")
axes[0].set_title("P&L by Edge Threshold")

# Win rate vs threshold
axes[1].plot(sens_df["threshold"], sens_df["win_rate"], "o-", color="green", linewidth=2)
axes[1].axhline(y=0.5, color="red", linestyle="--", label="Breakeven")
axes[1].set_xlabel("Edge Threshold")
axes[1].set_ylabel("Win Rate")
axes[1].set_title("Win Rate by Edge Threshold")
axes[1].legend()

# Sharpe vs threshold
axes[2].plot(sens_df["threshold"], sens_df["sharpe"], "o-", color="purple", linewidth=2)
axes[2].axhline(y=0, color="red", linestyle="--")
axes[2].set_xlabel("Edge Threshold")
axes[2].set_ylabel("Sharpe Ratio")
axes[2].set_title("Sharpe Ratio by Edge Threshold")

plt.tight_layout()
plt.show()

## 6. Regime Detection

Markets go through different regimes (high-scoring, low-scoring, high-variance, etc.). Testing across regimes shows if your strategy is robust.

In [ ]:
# Define synthetic regime periods
regime_periods = [
    RegimePeriod(start_date="0", end_date="60", regime_type="normal", description="Normal scoring"),
    RegimePeriod(start_date="61", end_date="120", regime_type="high_variance", description="High variance period"),
    RegimePeriod(start_date="121", end_date="200", regime_type="efficient", description="Efficient market"),
]

# Run walk-forward with regime awareness
regime_results = wfb.run_with_regimes(
    model_factory=lambda: GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
    regimes=regime_periods,
)

print(f"Regime-aware results: {regime_results.total_folds} folds")
print(f"\nRegime-specific metrics:")
for regime, metrics in regime_results.regime_metrics.items():
    print(f"\n  {regime}:")
    for k, v in metrics.items():
        if isinstance(v, (int, float)):
            print(f"    {k}: {v:.4f}")
        else:
            print(f"    {k}: {v}")

print(f"\nCross-regime comparison:")
for comp_key, comp_val in regime_results.cross_regime_comparison.items():
    print(f"  {comp_key}: {comp_val}")

In [ ]:
# Visualize regime performance
regime_names = list(regime_results.regime_metrics.keys())
regime_ll = []
regime_acc = []
for name in regime_names:
    m = regime_results.regime_metrics[name]
    regime_ll.append(m.get("log_loss_mean", 0))
    regime_acc.append(m.get("accuracy_mean", 0))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["steelblue", "coral", "forestgreen"]
axes[0].bar(regime_names, regime_ll, color=colors[:len(regime_names)], alpha=0.7)
axes[0].set_ylabel("Mean Log Loss")
axes[0].set_title("Log Loss by Market Regime (lower is better)")

axes[1].bar(regime_names, regime_acc, color=colors[:len(regime_names)], alpha=0.7)
axes[1].axhline(y=0.5, color="red", linestyle="--", label="Random")
axes[1].set_ylabel("Mean Accuracy")
axes[1].set_title("Accuracy by Market Regime")
axes[1].legend()

plt.tight_layout()
plt.show()

## Key Takeaway

> **Walk-forward validation is the gold standard; never trust a simple backtest.**

1. A simple backtest can look great due to overfitting or look-ahead bias
2. Walk-forward trains only on *past* data, mimicking real deployment
3. Sensitivity analysis finds optimal parameters without manual tuning
4. Regime detection shows whether your strategy works in *all* market conditions
5. Always check: ROI, Sharpe ratio, max drawdown, and profit factor together
6. A strategy that only works in one regime is fragile — diversify your approach